# Workshop 4: EDA Pipeline (Titanic)

Notebook นี้สำหรับฝึก Exploratory Data Analysis (EDA) จาก Titanic dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toche7/SlideAIDATADGA/blob/main/slides/workshop-04-eda-pipeline.ipynb)

## Learning Objectives
- โหลดข้อมูลที่ผ่าน Data Preparation แล้ว
- สำรวจโครงสร้างข้อมูลและสถิติเบื้องต้น
- วิเคราะห์ความสัมพันธ์ที่เกี่ยวข้องกับการรอดชีวิต
- สรุป Insight ที่นำไปใช้ต่อในงานวิเคราะห์

In [ ]:
# 1) Import libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

In [ ]:
# 2) Load cleaned Titanic data from Workshop 3 (fallback included)
clean_path = '/content/workshop3_titanic_clean.csv'
fallback_url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'

if os.path.exists(clean_path):
    df = pd.read_csv(clean_path)
    source_name = 'workshop3_titanic_clean.csv'
else:
    df = pd.read_csv(fallback_url)
    source_name = 'fallback Titanic CSV'

    # Minimal cleaning for standalone run
    if {'age', 'sex', 'Pclass'}.issubset(df.columns):
        df['age'] = df.groupby(['sex', 'Pclass'])['age'].transform(lambda s: s.fillna(s.median()))

    if 'Embarked' in df.columns:
        mode_embarked = df['Embarked'].mode(dropna=True)
        if not mode_embarked.empty:
            df['Embarked'] = df['Embarked'].fillna(mode_embarked.iloc[0])

    if 'Fare' in df.columns:
        df['Fare'] = pd.to_numeric(df['Fare'], errors='coerce').fillna(df['Fare'].median())

print('Loaded from:', source_name)
print('Shape:', df.shape)
df.head()

In [ ]:
# 3) Standardize column names (support both seaborn and Kaggle-style schema)
rename_map = {
    'Survived': 'survived',
    'Pclass': 'pclass',
    'Sex': 'sex',
    'Age': 'age',
    'SibSp': 'sibsp',
    'Parch': 'parch',
    'Fare': 'fare',
    'Embarked': 'embarked',
    'PassengerId': 'passenger_id',
    'Name': 'name',
    'Ticket': 'ticket',
    'Cabin': 'cabin'
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

print('Columns:', list(df.columns))
df.head(3)

In [ ]:
# 4) Quick profile
display(df.info())
display(df.describe(include='all').T)

missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
display(pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct}).head(15))

In [ ]:
# 5) Univariate EDA
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

if 'age' in df.columns:
    sns.histplot(df['age'], bins=30, kde=True, ax=axes[0])
    axes[0].set_title('Age Distribution')

if 'fare' in df.columns:
    sns.histplot(df['fare'], bins=30, kde=True, ax=axes[1])
    axes[1].set_title('Fare Distribution')

if 'pclass' in df.columns:
    sns.countplot(data=df, x='pclass', ax=axes[2])
    axes[2].set_title('Passenger Class Count')

plt.tight_layout()
plt.show()

In [ ]:
# 6) Bivariate EDA: survival by sex and class
if {'survived', 'sex'}.issubset(df.columns):
    surv_by_sex = df.groupby('sex', as_index=False)['survived'].mean().sort_values('survived', ascending=False)
    display(surv_by_sex)

    plt.figure(figsize=(6, 4))
    sns.barplot(data=surv_by_sex, x='sex', y='survived')
    plt.title('Survival Rate by Sex')
    plt.ylim(0, 1)
    plt.show()

if {'survived', 'pclass'}.issubset(df.columns):
    surv_by_class = df.groupby('pclass', as_index=False)['survived'].mean().sort_values('pclass')
    display(surv_by_class)

    plt.figure(figsize=(6, 4))
    sns.barplot(data=surv_by_class, x='pclass', y='survived')
    plt.title('Survival Rate by Passenger Class')
    plt.ylim(0, 1)
    plt.show()

In [ ]:
# 7) Correlation analysis (numeric columns)
num_df = df.select_dtypes(include=[np.number]).copy()
corr = num_df.corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='Blues', fmt='.2f')
plt.title('Correlation Heatmap (Numeric Features)')
plt.show()

In [ ]:
# 8) Tidy summary for reporting
summary = pd.DataFrame()

if {'survived', 'pclass', 'sex'}.issubset(df.columns):
    summary = (
        df.groupby(['pclass', 'sex'], as_index=False)
          .agg(
              passengers=('survived', 'size'),
              survival_rate=('survived', 'mean'),
              avg_age=('age', 'mean') if 'age' in df.columns else ('survived', 'size'),
              avg_fare=('fare', 'mean') if 'fare' in df.columns else ('survived', 'size')
          )
    )

if not summary.empty:
    summary['survival_rate'] = summary['survival_rate'].round(3)
    if 'avg_age' in summary.columns:
        summary['avg_age'] = summary['avg_age'].round(2)
    if 'avg_fare' in summary.columns:
        summary['avg_fare'] = summary['avg_fare'].round(2)

display(summary)

In [ ]:
# 9) Export outputs
df.to_csv('workshop4_titanic_eda_base.csv', index=False)
if 'summary' in globals() and len(summary) > 0:
    summary.to_csv('workshop4_titanic_eda_summary.csv', index=False)
    print('Saved: workshop4_titanic_eda_summary.csv')
print('Saved: workshop4_titanic_eda_base.csv')

## Reflection Questions
1. ปัจจัยใดดูมีผลต่อการรอดชีวิตมากที่สุดระหว่าง sex, pclass, fare, age?
2. มีความเสี่ยงจากข้อมูลไม่สมบูรณ์หรือความเอนเอียงอะไรบ้าง?
3. ถ้าต้องวิเคราะห์ต่อเชิงนโยบาย ควรเพิ่มข้อมูลอะไรอีก?
4. เขียน Insight 3 ข้อในรูปแบบ bullet สำหรับนำเสนอผู้บริหาร